# 04 — International EDA and conservative feature engineering (TRAIN ONLY)

This study is optional, separate from the U.S.-primary challenge deliverable. The international subset includes only **named-location evidence**, not all campaigns excluded by the U.S. filter. Small per-country samples are expected; use counts and **95% Wilson intervals** instead of ranking noisy country rates.

**Launch-time policy:** Exclude updates, comments, pledged amounts, backers, funded percentage, date-derived features of unverified meaning, title flags and reward-string features. The code fits country-based funding-goal reference medians **only on training data (or the inner CV training folds when modeling)**. It never treats country as state. Historical currencies may differ, so goal comparisons across countries are a limitation and comparisons within named regions remain exploratory.

In [1]:
from pathlib import Path
import sys
HERE = Path.cwd()
PROJECT_ROOT = next((folder for folder in (HERE, *HERE.parents)
                     if (folder / "utils" / "preprocessing.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter from inside the 1C repository")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
GLOBAL_DIR = PROJECT_ROOT / "data" / "Processed-Data"
INT_DIR = GLOBAL_DIR / "international_only"
CLEAN_GLOBAL = GLOBAL_DIR / "DSI_kickstarterscrape_dataset_cleaned.csv"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from utils.features import rate_table
from utils.preprocessing import drop_leaks
from utils.international_scope import classify_locations
from utils.international_features import InternationalFeatureBuilder
for path in [INT_DIR / "DSI_kickstarterscrape_dataset_cleaned.csv", INT_DIR / "train_project_ids.csv"]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}; complete international notebooks 02 and 03 first")
clean = pd.read_csv(INT_DIR / "DSI_kickstarterscrape_dataset_cleaned.csv")
train_ids = pd.read_csv(INT_DIR / "train_project_ids.csv")
training = clean.set_index("project id", drop=False).loc[train_ids["project id"]].reset_index(drop=True)
training["target"] = train_ids["target"].astype(int).to_numpy()
assert np.array_equal(training["status"].eq("successful").astype(int), training["target"])
training["country_region"] = classify_locations(training["location"])["country_region"].to_numpy()
print("Named-international TRAINING campaigns:", len(training))
print("Training success rate:", f"{training['target'].mean():.1%}" if len(training) else "no training rows")
print("Represented country/region names:", training["country_region"].nunique())

Named-international TRAINING campaigns: 1427
Training success rate: 61.0%
Represented country/region names: 125


In [2]:
# Start with geography sample sizes and explicitly show insufficient-support groups.
country_counts = training["country_region"].value_counts(dropna=False)
display(country_counts.head(25).to_frame("n"))
print("Country/region groups with < 25 labeled training campaigns:", int((country_counts < 25).sum()))
if len(training) >= 20:
    country_rates = rate_table(training, "country_region", min_n=25).sort_values("n", ascending=False)
    display(country_rates.head(25).round(3))
    if len(country_rates):
        fig, ax = plt.subplots(figsize=(9, max(3, 0.32 * min(12, len(country_rates)))))
        top = country_rates.head(12).sort_values("rate")
        ax.errorbar(top["rate"], top.index,
                    xerr=[top["rate"] - top["ci_lo"], top["ci_hi"] - top["rate"]], fmt="o", capsize=2)
        ax.axvline(training["target"].mean(), linestyle="--", color="gray")
        ax.set_xlim(0, 1); ax.set_xlabel("Historical success rate (95% Wilson interval)")
        ax.set_title("Country/region groups with at least 25 training campaigns")
        plt.tight_layout()
else:
    print("Not enough training rows for meaningful rate plots")

,n
country_region,
Canada,189
United Kingdom,124
Germany,67
Mexico,54
France,54
India,53
Italy,47
China,38
Australia,33


Country/region groups with < 25 labeled training campaigns: 113


,n,successes,rate,ci_lo,ci_hi,vs_base
country_region,,,,,,
Canada,189,117,0.619,0.548,0.685,0.009
United Kingdom,124,80,0.645,0.558,0.724,0.035
Germany,67,40,0.597,0.477,0.706,-0.013
France,54,35,0.648,0.515,0.762,0.038
Mexico,54,29,0.537,0.406,0.663,-0.073
India,53,30,0.566,0.433,0.690,-0.044
Italy,47,26,0.553,0.412,0.686,-0.056
China,38,21,0.553,0.397,0.699,-0.057
Australia,33,21,0.636,0.466,0.778,0.027


In [3]:
# Country composition may drive category associations: always display n and interval.
if len(training) >= 20:
    display(rate_table(training, "category", min_n=20).sort_values("n", ascending=False).head(20).round(3))
    display(rate_table(training, "subcategory", min_n=15).sort_values("n", ascending=False).head(20).round(3))
    duration = pd.to_numeric(training["duration"], errors="coerce")
    duration_bin = pd.cut(duration, bins=[0, 15, 30, 45, 60, 90, float("inf")],
        labels=["(0, 15]d", "(15, 30]d", "(30, 45]d", "(45, 60]d", "(60, 90]d", ">90d"])
    display(rate_table(training, duration_bin, min_n=15).round(3))
    # Within named region only: still subject to undocumented goal currency.
    medians = training.groupby("country_region")["goal"].agg(n="size", median_goal="median")
    print("Goal medians are not reliably comparable across countries until currency is verified")
    display(medians.sort_values("n", ascending=False).head(20).round(2))

,n,successes,rate,ci_lo,ci_hi,vs_base
category,,,,,,
Film & Video,539,294,0.545,0.503,0.587,-0.064
Art,211,154,0.730,0.666,0.785,0.120
Music,164,110,0.671,0.596,0.738,0.061
Photography,133,82,0.617,0.532,0.695,0.007
Publishing,121,62,0.512,0.424,0.600,-0.097
Theater,63,53,0.841,0.732,0.911,0.232
Design,53,35,0.660,0.526,0.773,0.051
Games,43,20,0.465,0.325,0.611,-0.145
Technology,30,16,0.533,0.361,0.698,-0.076


,n,successes,rate,ci_lo,ci_hi,vs_base
subcategory,,,,,,
Documentary,281,142,0.505,0.447,0.563,-0.104
Photography,133,82,0.617,0.532,0.695,0.007
Short Film,87,62,0.713,0.610,0.797,0.103
Film & Video,81,46,0.568,0.459,0.670,-0.042
Art,66,55,0.833,0.726,0.904,0.224
Theater,63,53,0.841,0.732,0.911,0.232
Narrative Film,52,21,0.404,0.282,0.539,-0.206
Music,47,34,0.723,0.582,0.831,0.114
Painting,40,29,0.725,0.572,0.839,0.115


,n,successes,rate,ci_lo,ci_hi,vs_base
duration,,,,,,
"(0, 15]d",56,34,0.607,0.476,0.724,-0.003
"(15, 30]d",546,357,0.654,0.613,0.693,0.044
"(30, 45]d",398,245,0.616,0.567,0.662,0.006
"(45, 60]d",267,154,0.577,0.517,0.635,-0.033
"(60, 90]d",147,79,0.537,0.457,0.616,-0.072


Goal medians are not reliably comparable across countries until currency is verified


,n,median_goal
country_region,,
Canada,189,5000.0
United Kingdom,124,4500.0
Germany,67,3500.0
France,54,3250.0
Mexico,54,5000.0
India,53,5000.0
Italy,47,5000.0
China,38,5000.0
Australia,33,10000.0


In [4]:
# Feature engineering is descriptive here, but the same builder will be
# fitted independently inside every CV fold in notebook 05.
raw_train = drop_leaks(training.drop(columns=["status", "target", "country_region"]))
FEATURE_POLICY = dict(include_timing=False, include_flags=False,
                      include_reward=False, exclude_renamed=True, min_group_n=10)
builder = InternationalFeatureBuilder(**FEATURE_POLICY)
features = builder.fit_transform(raw_train)
assert "country_region" in features.columns and "state" not in features.columns
assert "goal_vs_country" in features and "goal_vs_country_cat" in features
assert "goal_vs_subcat" not in features and "goal_vs_cat" not in features
assert "backers_needed" not in features and "launch_month_sin" not in features
assert "renamed" not in features
print("Conservative feature columns:", features.columns.tolist())
display(features.describe(include="all").T)
print("Numeric feature missingness:")
display(features.select_dtypes(include="number").isna().sum().sort_values(ascending=False))

Conservative feature columns: ['log_goal', 'goal_per_day', 'duration', 'category', 'subcategory', 'goal_vs_country', 'goal_vs_country_cat', 'country_region']


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
log_goal,1427.0,NaN,NaN,NaN,8.494215,1.181731,4.615121,7.779668,8.517393,9.21044,13.997833
goal_per_day,1427.0,NaN,NaN,NaN,4.872848,1.126288,1.181133,4.167415,4.828814,5.55398,9.886959
duration,1427.0,NaN,NaN,NaN,39.956111,17.647002,3.0,30.0,32.44,50.0,91.22
category,1427,13,Film & Video,539,NaN,NaN,NaN,NaN,NaN,NaN,NaN
subcategory,1427,49,Documentary,281,NaN,NaN,NaN,NaN,NaN,NaN,NaN
goal_vs_country,1427.0,NaN,NaN,NaN,0.05685,1.160812,-3.902273,-0.692947,0.0,0.693047,5.585778
goal_vs_country_cat,1427.0,NaN,NaN,NaN,0.057437,1.128199,-3.902273,-0.640372,0.0,0.693047,5.48044
country_region,1427,125,Canada,189,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Numeric feature missingness:


log_goal               0
goal_per_day           0
duration               0
goal_vs_country        0
goal_vs_country_cat    0
dtype: int64

## Interpretation and handoff to 05

- Small country cells and multiple comparisons mean apparent success-rate differences can easily be noise. **Country/region is not a protected-characteristic classifier, a verified account country, or a causal explanation.**
- `goal_vs_country` and `goal_vs_country_cat` use **training-fold** goal medians. Their historical comparability still depends on consistent goal currency definitions; do not claim international affordability or funding equivalence.
- Geographic location, category, and duration might have been edited between launch and scrape; provenance cannot guarantee they are historical snapshots.
- No post-launch activity variables enter models; the unknown meaning of `funded date` remains a blocker for temporal features.
- None of these exploratory training-only group rates may be used as empirical causal recommendations to change categories or location.
- Proceed to `05_modeling.ipynb` for a modest baseline check; do not tune or open the international test labels. Final U.S. selection and testing still take priority.